In [1]:
import ssl
import urllib.request
import urllib.parse
from bs4 import BeautifulSoup
import pandas as pd
import time
import json
from datetime import datetime
from pathlib import Path

# ==============================================================================
# 1. CONFIGURACIÓ DE L'ENTORN I BYPASS DE SEGURETAT
# ==============================================================================
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
context = ssl._create_unverified_context()

OUTPUT_DIR = Path("../../data/raw/championsxip")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "DF_CHAMPIONSXIP_SUCIO.csv"
CHECKPOINT_FILE = OUTPUT_DIR / "championsxip_checkpoint.json"

# Pausar i reprendre: amb RESUME=True es reutilitza el CSV existent i se
# salten els períodes mensuals que ja van quedar tancats (data_fi anterior
# a avui) en una execució prèvia — només es tornen a escanejar el mes en
# curs (per si s'hi han afegit curses noves) i els mesos futurs. Per
# començar de zero, posa RESUME = False o esborra CHECKPOINT_FILE.
RESUME = True

avui = datetime.now().date()

# Generem la llista de mesos des de 2021 fins a l'any actual (dinàmic:
# ja no cal actualitzar 'fi_any' a mà cada any)
inici_any = 2021
fi_any = avui.year
periodes = []

for any_num in range(inici_any, fi_any + 1):
    for mes in range(1, 13):
        if any_num == fi_any and mes > avui.month:
            break
        data_inici = f"{any_num}-{mes:02d}-01"
        if mes == 12:
            data_fi = f"{any_num}-12-31"
        else:
            data_fi = f"{any_num}-{mes+1:02d}-01"
        periodes.append((data_inici, data_fi))

# Carreguem el que ja teníem d'execucions anteriors
totes_les_curses = []
periodes_completats = set()
if RESUME and OUTPUT_CSV.exists():
    df_existent = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    totes_les_curses = df_existent.to_dict('records')
    print(f"Resume: es reprèn amb {len(totes_les_curses)} registres ja desats a {OUTPUT_CSV}")
if RESUME and CHECKPOINT_FILE.exists():
    with open(CHECKPOINT_FILE, encoding='utf-8') as f:
        periodes_completats = set(json.load(f))
    print(f"Resume: {len(periodes_completats)} períodes mensuals ja escanejats es saltaran.")

print(f"Començant l'extracció per a {len(periodes)} períodes mensuals...")

# ==============================================================================
# 2. BUCLE PRINCIPAL D'EXTRACCIÓ
# ==============================================================================
for data_inici, data_fi in periodes:
    clau_periode = data_inici[:7]  # "YYYY-MM"
    periode_tancat = datetime.strptime(data_fi, "%Y-%m-%d").date() <= avui

    if RESUME and periode_tancat and clau_periode in periodes_completats:
        continue

    print(f"🔍 Escanejant el període: {data_inici} fins a {data_fi}...")

    pag = 1
    pagines_buides_consecutives = 0
    MAX_PAGINES_BUIDES = 2
    curses_del_periode = []

    while True:
        url_base = "https://championsxip.cat/cursas"
        params = {
            'q[data_gteq]': data_inici,
            'q[data_lteq]': data_fi,
            'q[run_type_cont_any][]': ['running', 'trail', 'caminada'],
            'curses_view_type': 'small',
            'result_preset': 'all',
            'page': pag
        }

        url_encoded = url_base + "?" + urllib.parse.urlencode(params, doseq=True)

        try:
            req = urllib.request.Request(url_encoded, headers=headers)
            with urllib.request.urlopen(req, context=context) as response:
                html = response.read()

            soup = BeautifulSoup(html, 'html.parser')

            # Busquem els articles utilitzant la classe exacta de l'HTML que has passat
            articles_curses = soup.select('article.small-cursa-box.cursa')

            if not articles_curses:
                pagines_buides_consecutives += 1
                if pagines_buides_consecutives >= MAX_PAGINES_BUIDES:
                    break
                pag += 1
                continue

            pagines_buides_consecutives = 0

            for article in articles_curses:
                # 1. Extreure Data
                date_elem = article.select_one('.date-format span')
                data_cursa = date_elem.get_text(strip=True) if date_elem else ""

                # 2. Extreure Nom
                title_elem = article.select_one('.cursa-info .title a')
                nom_cursa = title_elem.get_text(strip=True) if title_elem else ""

                # 3. Extreure Modalitat General (running, trail, caminada)
                type_elem = article.select_one('.cursa-info .cursa-type')
                modalitat = ""
                if type_elem:
                    # Traiem el text netejant icones o espais en blanc residuals
                    modalitat = type_elem.get_text(strip=True).replace('v', '').replace('c', '').strip()

                # 4. Processar les Sub-curses / Distàncies (Scraping de Nivell 2)
                subcurses_links = article.select('.subcursas a.main-link')

                # Si no té sub-curses, guardem el registre base
                if not subcurses_links:
                    curses_del_periode.append({
                        'Nom_Cursa': nom_cursa,
                        'Data': data_cursa,
                        'Modalitat': modalitat,
                        'Distancia_Subcursa': "Única / No especificada",
                        'Finishers_Homes': 0,
                        'Finishers_Dones': 0,
                        'URL_Resultats': ""
                    })
                    continue

                # Si en té, entrem a cadascuna per comptar els finishers
                for link in subcurses_links:
                    nom_distancia = link.get_text(strip=True) # Ex: "13K", "5K"
                    href_resultats = link.get('href', '')

                    url_resultats_completa = ""
                    finishers_homes = 0
                    finishers_dones = 0

                    if href_resultats:
                        if not href_resultats.startswith('http'):
                            url_resultats_completa = "https://championsxip.cat" + href_resultats
                        else:
                            url_resultats_completa = href_resultats

                        # Anem a buscar la classificació d'aquesta distància específica
                        try:
                            time.sleep(0.4) # Control de cortesia
                            req_detall = urllib.request.Request(url_resultats_completa, headers=headers)
                            with urllib.request.urlopen(req_detall, context=context) as resp_detall:
                                html_detall = resp_detall.read()

                            soup_detall = BeautifulSoup(html_detall, 'html.parser')
                            taula_resultats = soup_detall.find('table')

                            if taula_resultats:
                                files_taula = taula_resultats.find_all('tr')[1:]
                                for f_res in files_taula:
                                    columnes = [c.get_text(strip=True).upper() for c in f_res.find_all('td')]
                                    if columnes:
                                        # Heurística de cerca de gènere en brut a les columnes
                                        if any(g in columnes for g in ['M', 'H', 'MASCULÍ', 'MASCULINO', 'HOMES']):
                                            finishers_homes += 1
                                        elif any(g in columnes for g in ['F', 'D', 'FEMENÍ', 'FEMENINO', 'DONES']):
                                            finishers_dones += 1
                        except Exception:
                            pass # Si falla una taula concreta, continuem

                    # Guardem tota la informació estructurada per distància
                    curses_del_periode.append({
                        'Nom_Cursa': nom_cursa,
                        'Data': data_cursa,
                        'Modalitat': modalitat,
                        'Distancia_Subcursa': nom_distancia,
                        'Finishers_Homes': finishers_homes,
                        'Finishers_Dones': finishers_dones,
                        'URL_Resultats': url_resultats_completa
                    })

            print(f"   ↳ Pàgina {pag}: Extretes {len(articles_curses)} curses.")
            pag += 1
            time.sleep(0.8)

        except Exception as e:
            print(f"🚨 Error crític a la pàgina {pag} del període {data_inici}: {e}")
            break

    totes_les_curses.extend(curses_del_periode)

    # Desem progrés parcial després de cada període, per no perdre feina
    # si l'execució s'interromp a mig camí, i marquem el període com fet
    # només si ja ha tancat (així el mes en curs sempre es torna a mirar).
    df_parcial = pd.DataFrame(totes_les_curses).drop_duplicates(
        subset=['Nom_Cursa', 'Data', 'Distancia_Subcursa', 'URL_Resultats']
    )
    df_parcial.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

    if periode_tancat:
        periodes_completats.add(clau_periode)
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump(sorted(periodes_completats), f)

# ==============================================================================
# 3. EXPORTACIÓ FINAL
# ==============================================================================
df_raw = pd.DataFrame(totes_les_curses).drop_duplicates(
    subset=['Nom_Cursa', 'Data', 'Distancia_Subcursa', 'URL_Resultats']
)
df_raw.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n Extracció finalitzada. {len(df_raw)} registres en total a: {OUTPUT_CSV}")
print(df_raw.head())

Resume: es reprèn amb 2727 registres ja desats a ../../data/raw/championsxip/DF_CHAMPIONSXIP_SUCIO.csv
Començant l'extracció per a 69 períodes mensuals...
🔍 Escanejant el període: 2021-01-01 fins a 2021-02-01...
🚨 Error crític a la pàgina 1 del període 2021-01-01: <urlopen error [Errno 8] nodename nor servname provided, or not known>
🔍 Escanejant el període: 2021-02-01 fins a 2021-03-01...
🚨 Error crític a la pàgina 1 del període 2021-02-01: <urlopen error [Errno 8] nodename nor servname provided, or not known>
🔍 Escanejant el període: 2021-03-01 fins a 2021-04-01...
🚨 Error crític a la pàgina 1 del període 2021-03-01: <urlopen error [Errno 8] nodename nor servname provided, or not known>
🔍 Escanejant el període: 2021-04-01 fins a 2021-05-01...
🚨 Error crític a la pàgina 1 del període 2021-04-01: <urlopen error [Errno 8] nodename nor servname provided, or not known>
🔍 Escanejant el període: 2021-05-01 fins a 2021-06-01...
🚨 Error crític a la pàgina 1 del període 2021-05-01: <urlopen er